### This book shows how we fill library, and here we will collect results 

What should our tensors include?
Features which describe size of catenary curves, transmissions and phase shift.

Подумаем от каких параметров меняется форма ячейки.
1. Мы можем изменять высоту ячейки H, но это не типичный параметр для вариации
2. Мы можем менять изгиб 'a', что влияет на скорость смещения центра. Мы можем изменять зависимость расстояния между верхней и нижней кривой. Сейчас это константа для одной формы. Если это не будет константой, то нужно будет менять способ расчета сечений.
Итого, получаем высота - не изменяемый параметр для всей библиотеки. y_center зависит а. Итого получаем 2 изменяемых параметра при текущей конфигурации.     

In [59]:
import meep as mp
import conf
import numpy as np

In [60]:
pml_layers = [mp.PML(thickness=conf.in_pml, direction=mp.X)]
cell = mp.Vector3(conf.cell_X, conf.cell_Y, 0)
symmetries = [mp.Mirror(mp.Y)]
geometry = [mp.Block(
    material=conf.in_material,
    size=mp.Vector3(conf.in_sub, mp.inf, conf.cell_Z),
    center=conf.sub_center,
)]
sources = [mp.Source(
    mp.ContinuousSource(conf.inc_fcen, fwidth=conf.inc_df),
    component=mp.Ez,
    center=conf.source_center,
    size=mp.Vector3(x=0, y=conf.cell_Y, z=conf.cell_Z),
)]

sim = mp.Simulation(
    cell_size=cell,
    boundary_layers=pml_layers,
    geometry=geometry,
    sources=sources,
    resolution=conf.resolution,
    symmetries = symmetries,
)
flux_obj = sim.add_flux(
            conf.inc_fcen, 0, 1, mp.FluxRegion(center=conf.flux_center, size=mp.Vector3(y=conf.cell_Y, z=conf.cell_Z))
)
sim.run(until=conf.steps) # run 1st part of simulation
input_flux = mp.get_fluxes(flux_obj) # saving flux from sub in input_flux
sim.reset_meep() # purge all adjustments of simulation

FloatProgress(value=0.0, description='0% done ', max=50.0)

In [61]:
step = 10
mode_tran = np.empty(step)
mode_phase = np.empty(step)
size_block = [None] * step  # Список для объектов mp.Vector3
min_width, max_width = 0.3, 2
min_a, max_a = 0.5, 3

# Параметры цепной линии
a = np.linspace(min_a, max_a, step)  # параметр формы цепной линии
width = np.linspace(min_width, max_width, step) # параметр ширины цепной линии
x_max = 1  # диапазон x
x_min = -x_max
number_sections = 1
x_sections = np.linspace(0, x_max,  number_sections, endpoint=False) # В этих точках будем вычислять значения для y1 and y2
geometry.append(mp.Block(
    material=conf.at_material,
    size=mp.Vector3(0,0,0),
    center=mp.Vector3(0,0,0),
))

In [62]:
width[4]

1.0555555555555556

In [64]:
mp.Verbosity(0)
for j in range(step):
    size_block[j] = mp.Vector3(x=conf.at_hei, y=width[j])
    print(width[j])
    for i in range(number_sections): #фактически номер секции сейчас не используется, и это просто заготовка. Внутри код где-то это не учитывает
        y_center = a[i] * np.cosh(x_sections[i] / a[i]) - a[i] 
        geometry[1].size = size_block[j]
        geometry[1].center =mp.Vector3(y=y_center)
        
        sim = mp.Simulation(
            cell_size=cell,
            boundary_layers=pml_layers,
            geometry=geometry,
            sources=sources,
            resolution=conf.resolution,
        )
        flux_obj = sim.add_flux(
            conf.inc_fcen, 0, 1, mp.FluxRegion(
                center=mp.Vector3(x=conf.flux_center[0]), 
                size=mp.Vector3(y=conf.cell_Y)
            )
        )
        sim.run(until=conf.steps)
        
        res = sim.get_eigenmode_coefficients(
            flux_obj, [1], eig_parity=mp.ODD_Z + mp.EVEN_Y
        )
        coeffs = res.alpha
        
        mode_tran[j] = abs(coeffs[0, 0, 0]) ** 2 / input_flux[0]
        mode_phase[j] = np.angle(coeffs[0, 0, 0])
        if mode_phase[j] > 0:
            mode_phase[j] -= 2 * np.pi
        
        
        '''
        %matplotlib inline
        plt.figure(dpi=100)
        sim.plot2D(fields=mp.Ez)
        plt.show()
        sim.reset_meep() # purge all adjustments of simulation
        '''

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

FloatProgress(value=0.0, description='0% done ', max=50.0)

In [65]:
mode_phase, mode_tran

(array([-4.45787766, -4.33617559, -4.19341337, -4.00860138, -3.78548023,
        -3.57090017, -3.40019582, -3.29147181, -3.24745221, -3.25395803]),
 array([0.34706167, 0.22691554, 0.15017107, 0.10382022, 0.0799259 ,
        0.07296629, 0.07686089, 0.08835868, 0.10578451, 0.1296801 ]))

In [66]:
width

array([0.3       , 0.48888889, 0.67777778, 0.86666667, 1.05555556,
       1.24444444, 1.43333333, 1.62222222, 1.81111111, 2.        ])

In [67]:
a

array([0.5       , 0.77777778, 1.05555556, 1.33333333, 1.61111111,
       1.88888889, 2.16666667, 2.44444444, 2.72222222, 3.        ])

### Fine, now we want to save result as database

In [68]:
print(mode_tran.dtype)
print(mode_tran.ndim)

In [69]:
# Объединяем в один двумерный массив
data = np.column_stack((mode_phase, mode_tran, width, a))

# Сохраняем в файл в формате CSV
np.savetxt('vectors1.csv', data, delimiter=',', header='mode_phase,mode_tran,width,a', comments='', fmt='%.6f')

# Альтернатива: сохранить в бинарном формате .npy
np.save('vectors1.npy', data)

print("Данные успешно сохранены!")

In [70]:
data.ndim

2

In [71]:
data.size

40

In [72]:
data.dtype

dtype('float64')

Finally. Проверка завершена. Данные для обучения расчитываются и копятся корректно.
### TODO: Реализовать мультипоточность через параллелизм независимых процессов.

Теперь надо прикинуть, что делаем дальше. Мы имеем расчитанный набор данных. Но их еще нужно подготовить для обучения.
Нужно разделить на тест и трейн набор данных, а также разобраться с указанием тренируемых параметров. 